In [1]:

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Trained adapter path
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v7" 

print("Loading model...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

Loading model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.37it/s]


Tokenizer vocab size: 128256


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:693: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Model loaded successfully!


In [10]:
# Test the model with the CORRECT tokenizer (the one with special tokens)
def test_model(patient_id, query, max_length=1024):
    # Format exactly like training data - include the explicit instruction
    messages = [
        # {
        #     "role": "system",
        #     "content": "You are a compassionate mental health support assistant who responds in a warm, empathetic, and non-judgmental way, validating the user’s feelings, offering practical and evidence-based coping suggestions, encouraging professional help when appropriate, providing crisis resources only if there is self-harm or suicide risk, and tailoring hopeful guidance to the user’s specific situation."
        # },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            # temperature=0.01,              
            # top_p=0.95,                 
            # repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            
            
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

In [16]:
import textwrap

test_query = """I look very ugly"""
response = test_model("UID_123", test_query)

# Wrap text at 80 characters per line
print("Response\n")
print("\n".join(textwrap.wrap(response, width=100)))
# print("=" * 80)


Response

This is something that I think we have all heard before, at some point in our lives. If you are a
teenager or an adult, you probably remember hearing this from your parents or siblings a lot when
you looked in the mirror and noticed a zit, your newly gained hair, or if you were very skinny. But
it is especially common for teenagers to feel this way. What if I told you that most people are more
concerned about how you are doing as a person in your life, compared to what you look like on the
outside? What if I told you that people are most likely to remember your personality, your
achievements, and all the things that you have done to help other people, instead of what you looked
like?So, how do you change how you feel about the way that you look? This is a great question. Some
people decide to change their hairstyle, or maybe they get a little bit of makeup for a special
occasion. Other people like to join sports or clubs at school, so they can notice what other people
are int